In [3]:
import os
import math
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [4]:
CSV_PATH = "KO.csv"
RESULTS_CSV = "KO_quantile_lstm_grid_results.csv"
BEST_PRED_CSV ="KO_best_config_predictions.csv"

TARGET_COVERAGE = 0.95
LOOKBACK_GRID = [30, 60, 90, 120]
LAG_GRID = [1, 3, 5]
LAYER_GRID = [1, 2]
HIDDEN_GRID = [64, 128]
QUANTILE_GRID = [(0.05, 0.95), (0.025, 0.975), (0.01, 0.99)]


In [5]:
MODEL_KIND = "LSTM"

In [6]:
# Change this once to switch the core recurrent architecture everywhere.
# Valid options: "LSTM", "GRU", "RNN", "BiLSTM"


# Optional architecture comparison after the best LSTM configuration is selected.
ARCHITECTURES_TO_COMPARE = ["LSTM", "BiLSTM", "GRU"]

RANDOM_SEED = 42
BATCH_SIZE = 256
MAX_EPOCHS = 40
PATIENCE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
DROPOUT = 0.10
CALIBRATION_GRID = np.linspace(0.60, 2.50, 80)
BOOTSTRAP_REPS = 300

In [7]:
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
if use_amp and hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if use_amp:
    torch.backends.cudnn.benchmark = True

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
print(f"Using device: {device}")

Using device: cuda


In [9]:
def load_and_prep_data(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()

    # Robust date handling for timezone-aware and timezone-naive strings.
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_convert(None)
    df = df.sort_values("Date").reset_index(drop=True)

    # Vectorized price dynamics.
    df["log_return"] = np.log(df["Close"]).diff()
    df["volatility"] = df["log_return"].rolling(window=5, min_periods=5).std()
    df["hl_range"] = (df["High"] - df["Low"]) / df["Close"]

    # Semantic features.
    df["sentiment_signed"] = 2.0 * df["Scaled_sentiment"] - 1.0
    df["sentiment_active"] = df["sentiment_signed"] * df["News_flag"]
    df["sentiment_impact"] = df["sentiment_active"] * df["log_return"].shift(1)
    df["sentiment_impact_raw"] = df["sentiment_signed"] * df["log_return"].shift(1)

    # One-step-ahead regression target.
    df["target"] = df["Close"].shift(-1)

    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return df

In [10]:
def add_lag_features(df: pd.DataFrame, base_cols, n_lags: int) -> pd.DataFrame:
    """
    Create lagged versions of selected columns.
    n_lags is the experimental 'lag' axis in the grid search.
    """
    out = df.copy()
    lag_blocks = []
    for k in range(1, n_lags + 1):
        block = out[base_cols].shift(k).add_suffix(f"_lag{k}")
        lag_blocks.append(block)
    if lag_blocks:
        out = pd.concat([out] + lag_blocks, axis=1)
    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


In [11]:
def create_sequences(df: pd.DataFrame, feature_cols, lookback: int):
    """
    Turn the tabular time series into supervised sequences.
    The model sees the previous 'lookback' rows and predicts next-day Close.
    """
    X = df[feature_cols].to_numpy(dtype=np.float32, copy=False)
    y = df["target"].to_numpy(dtype=np.float32, copy=False)
    dates = df["Date"].to_numpy()

    X_seq, y_seq, d_seq = [], [], []
    for i in range(lookback, len(df)):
        X_seq.append(X[i - lookback:i])
        y_seq.append(y[i])
        d_seq.append(dates[i])

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.float32),
        np.asarray(d_seq),
    )

In [12]:
def chronological_split(X_seq, y_seq, dates, train_frac=0.70, val_frac=0.15):
    """
    Time-series split with no shuffling.
    """
    n = len(X_seq)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    splits = {
        "train": (X_seq[:train_end], y_seq[:train_end], dates[:train_end]),
        "val": (X_seq[train_end:val_end], y_seq[train_end:val_end], dates[train_end:val_end]),
        "test": (X_seq[val_end:], y_seq[val_end:], dates[val_end:]),
    }
    return splits

In [13]:
def make_loader(X, y=None, batch_size=256, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    if y is None:
        ds = TensorDataset(X_tensor)
    else:
        y_tensor = torch.tensor(y, dtype=torch.float32)
        ds = TensorDataset(X_tensor, y_tensor)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )

# 3. MODEL DEFINITION

In [14]:
class RecurrentQuantileNet(nn.Module):
    """
    Change MODEL_KIND to switch architectures globally:
    - "LSTM"
    - "GRU"
    - "RNN"
    - "BiLSTM"
    """
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, model_kind: str = "LSTM", dropout: float = 0.1):
        super().__init__()
        self.model_kind = model_kind.lower()
        self.bidirectional = self.model_kind == "bilstm"

        if self.model_kind in {"lstm", "bilstm"}:
            rnn_cls = nn.LSTM
        elif self.model_kind == "gru":
            rnn_cls = nn.GRU
        elif self.model_kind == "rnn":
            rnn_cls = nn.RNN
        else:
            raise ValueError(f"Unsupported model_kind: {model_kind}")

        self.hidden_size = hidden_size
        self.out_dim = hidden_size * 2 if self.bidirectional else hidden_size

        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=self.bidirectional,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(self.out_dim),
            nn.Linear(self.out_dim, self.out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.out_dim, 1),
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

In [15]:
def pinball_loss(pred, target, q: float):
    """
    Quantile / pinball loss.
    """
    e = target - pred
    return torch.mean(torch.maximum((q - 1.0) * e, q * e))

In [16]:
@torch.no_grad()
def predict_numpy(model, X, batch_size=512):
    model.eval()
    preds = []
    loader = make_loader(X, None, batch_size=batch_size, shuffle=False)
    autocast_ctx = torch.cuda.amp.autocast if use_amp else nullcontext
    for (xb,) in loader:
        xb = xb.to(device, non_blocking=True)
        with autocast_ctx():
            out = model(xb)
        preds.append(out.detach().float().cpu().numpy())
    return np.concatenate(preds, axis=0)

In [17]:
def fit_one_quantile_model(X_train, y_train, X_val, y_val, q: float, input_size: int,
                           hidden_size: int, num_layers: int, model_kind: str,
                           lr: float = LR, weight_decay: float = WEIGHT_DECAY,
                           max_epochs: int = MAX_EPOCHS, patience: int = PATIENCE):
    model = RecurrentQuantileNet(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        model_kind=model_kind,
        dropout=DROPOUT,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    train_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = make_loader(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)

    best_state = None
    best_val = float("inf")
    stale = 0

    for epoch in range(max_epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.cuda.amp.autocast():
                    pred = model(xb)
                    loss = pinball_loss(pred, yb, q)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                pred = model(xb)
                loss = pinball_loss(pred, yb, q)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            train_losses.append(loss.detach().item())

        # Validation pinball loss
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                if use_amp:
                    with torch.cuda.amp.autocast():
                        pred = model(xb)
                        loss = pinball_loss(pred, yb, q)
                else:
                    pred = model(xb)
                    loss = pinball_loss(pred, yb, q)
                val_losses.append(loss.item())

        val_loss = float(np.mean(val_losses))
        scheduler.step(val_loss)

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val

In [18]:

def fit_quantile_pair(X_train, y_train, X_val, y_val, input_size, hidden_size, num_layers, model_kind, q_low, q_high):
    low_model, low_val_loss = fit_one_quantile_model(
        X_train, y_train, X_val, y_val, q=q_low,
        input_size=input_size, hidden_size=hidden_size,
        num_layers=num_layers, model_kind=model_kind
    )
    high_model, high_val_loss = fit_one_quantile_model(
        X_train, y_train, X_val, y_val, q=q_high,
        input_size=input_size, hidden_size=hidden_size,
        num_layers=num_layers, model_kind=model_kind
    )
    return low_model, high_model, low_val_loss, high_val_loss

In [19]:
def interval_metrics(y_true, low, high):
    low = np.asarray(low)
    high = np.asarray(high)
    y_true = np.asarray(y_true)
    coverage = np.mean((y_true >= low) & (y_true <= high))
    width = np.mean(high - low)
    midpoint = 0.5 * (low + high)
    mse_mid = mean_squared_error(y_true, midpoint)
    return coverage, width, mse_mid

In [20]:
def calibrate_symmetric_interval(y_val, low_val, high_val, target_coverage=TARGET_COVERAGE):
    """
    Symmetric scaling around the midpoint.
    We search the narrowest interval that reaches the target coverage on validation.
    """
    y_val = np.asarray(y_val)
    low_val = np.asarray(low_val)
    high_val = np.asarray(high_val)

    mid = 0.5 * (low_val + high_val)
    half = 0.5 * (high_val - low_val)
    half = np.maximum(half, 1e-8)

    best = None
    for s in CALIBRATION_GRID:
        low_s = mid - s * half
        high_s = mid + s * half
        cov = np.mean((y_val >= low_s) & (y_val <= high_s))
        width = np.mean(high_s - low_s)
        obj = (cov < target_coverage, abs(cov - target_coverage), width)
        if best is None or obj < best["obj"]:
            best = {
                "scale": float(s),
                "coverage": float(cov),
                "width": float(width),
                "obj": obj,
            }

    return best["scale"], best


In [21]:

def apply_interval_scale(low, high, scale: float):
    mid = 0.5 * (np.asarray(low) + np.asarray(high))
    half = 0.5 * (np.asarray(high) - np.asarray(low))
    low_s = mid - scale * half
    high_s = mid + scale * half
    return low_s, high_s

In [22]:
def bootstrap_ci(y_true, low, high, reps=BOOTSTRAP_REPS, seed=42):
    """
    Bootstrap uncertainty for error bars.
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    low = np.asarray(low)
    high = np.asarray(high)
    n = len(y_true)

    covs, widths = [], []
    for _ in range(reps):
        idx = rng.integers(0, n, size=n)
        cov, width, _ = interval_metrics(y_true[idx], low[idx], high[idx])
        covs.append(cov)
        widths.append(width)

    covs = np.asarray(covs)
    widths = np.asarray(widths)
    cov_ci = np.percentile(covs, [2.5, 97.5])
    width_ci = np.percentile(widths, [2.5, 97.5])
    return cov_ci, width_ci

In [23]:
df = load_and_prep_data(CSV_PATH)

base_features = ["Close", "log_return", "volatility", "hl_range"]
semantic_gated_features = base_features + ["News_flag", "sentiment_active", "sentiment_impact"]
semantic_ungated_features = base_features + ["News_flag", "sentiment_signed", "sentiment_impact_raw"]

feature_sets = {
    "Base_Price_Only": base_features,
    "Semantic_Gated": semantic_gated_features,
    "Semantic_Ungated": semantic_ungated_features,
}


In [ ]:
results = []
artifacts = {}

for feature_set_name, feature_cols in feature_sets.items():
    print(f"\n===== FEATURE SET: {feature_set_name} =====")
    for n_lags in LAG_GRID:
        lagged_df = add_lag_features(df, feature_cols, n_lags)

        lagged_feature_cols = feature_cols + [
            f"{col}_lag{k}"
            for col in feature_cols
            for k in range(1, n_lags + 1)
        ]

        for lookback in LOOKBACK_GRID:
            X_seq, y_seq, dates_seq = create_sequences(lagged_df, lagged_feature_cols, lookback)
            if len(X_seq) < 100:
                continue

            splits = chronological_split(X_seq, y_seq, dates_seq, train_frac=0.70, val_frac=0.15)
            X_train, y_train, d_train = splits["train"]
            X_val, y_val, d_val = splits["val"]
            X_test, y_test, d_test = splits["test"]

            # Standardize using training data only.
            x_scaler = StandardScaler()
            y_scaler = StandardScaler()

            X_train_2d = X_train.reshape(-1, X_train.shape[-1])
            X_val_2d = X_val.reshape(-1, X_val.shape[-1])
            X_test_2d = X_test.reshape(-1, X_test.shape[-1])

            x_scaler.fit(X_train_2d)
            y_scaler.fit(y_train.reshape(-1, 1))

            def transform_X(X):
                X2 = X.reshape(-1, X.shape[-1])
                Xs = x_scaler.transform(X2).reshape(X.shape)
                return Xs.astype(np.float32)

            X_train_s = transform_X(X_train)
            X_val_s = transform_X(X_val)
            X_test_s = transform_X(X_test)

            y_train_s = y_scaler.transform(y_train.reshape(-1, 1)).ravel().astype(np.float32)
            y_val_s = y_scaler.transform(y_val.reshape(-1, 1)).ravel().astype(np.float32)
            y_test_s = y_scaler.transform(y_test.reshape(-1, 1)).ravel().astype(np.float32)

            input_size = X_train_s.shape[-1]

            for num_layers in LAYER_GRID:
                for hidden_size in HIDDEN_GRID:
                    for q_low, q_high in QUANTILE_GRID:
                        q_label = f"{q_low:.3f}/{q_high:.3f}"
                        config_name = f"{feature_set_name}|L{lookback}|lag{n_lags}|{num_layers}x{hidden_size}|q{q_label}|{MODEL_KIND}"

                        print(f"Training {config_name}")

                        low_model, high_model, low_vloss, high_vloss = fit_quantile_pair(
                            X_train_s, y_train_s, X_val_s, y_val_s,
                            input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            model_kind=MODEL_KIND,
                            q_low=q_low,
                            q_high=q_high,
                        )

                        # Validation predictions in original price scale.
                        low_val_s = predict_numpy(low_model, X_val_s)
                        high_val_s = predict_numpy(high_model, X_val_s)
                        low_val = y_scaler.inverse_transform(low_val_s.reshape(-1, 1)).ravel()
                        high_val = y_scaler.inverse_transform(high_val_s.reshape(-1, 1)).ravel()

                        # Calibration scaling: smallest interval that reaches target coverage on validation.
                        scale, calib_info = calibrate_symmetric_interval(y_val, low_val, high_val, TARGET_COVERAGE)

                        # Test predictions in original price scale.
                        low_test_s = predict_numpy(low_model, X_test_s)
                        high_test_s = predict_numpy(high_model, X_test_s)
                        low_test = y_scaler.inverse_transform(low_test_s.reshape(-1, 1)).ravel()
                        high_test = y_scaler.inverse_transform(high_test_s.reshape(-1, 1)).ravel()

                        low_test_cal, high_test_cal = apply_interval_scale(low_test, high_test, scale)
                        low_val_cal, high_val_cal = apply_interval_scale(low_val, high_val, scale)

                        test_cov, test_width, test_mse = interval_metrics(y_test, low_test_cal, high_test_cal)
                        val_cov, val_width, _ = interval_metrics(y_val, low_val_cal, high_val_cal)

                        result_row = {
                            "model_name": "LSTM",
                            "feature_set": feature_set_name,
                            "lookback": lookback,
                            "lags": n_lags,
                            "num_layers": num_layers,
                            "hidden_size": hidden_size,
                            "q_low": q_low,
                            "q_high": q_high,
                            "quantile_label": q_label,
                            "target_coverage": TARGET_COVERAGE,
                            "calibration_scale": scale,
                            "val_coverage": val_cov,
                            "val_width": val_width,
                            "test_coverage": test_cov,
                            "test_width": test_width,
                            "test_mse_mid": test_mse,
                            "val_pinball_low": low_vloss,
                            "val_pinball_high": high_vloss,
                            "n_train": len(X_train_s),
                            "n_val": len(X_val_s),
                            "n_test": len(X_test_s),
                        }
                        results.append(result_row)

                        artifacts[config_name] = {
                            "dates_test": d_test,
                            "y_test": y_test,
                            "low_test": low_test_cal,
                            "high_test": high_test_cal,
                            "mid_test": 0.5 * (low_test_cal + high_test_cal),
                            "feature_set": feature_set_name,
                            "lookback": lookback,
                            "lags": n_lags,
                            "num_layers": num_layers,
                            "hidden_size": hidden_size,
                            "q_label": q_label,
                            "scale": scale,
                        }

results_df = pd.DataFrame(results)

# Rank by coverage proximity first, then narrower intervals, then lower midpoint error.
results_df["coverage_gap"] = (results_df["test_coverage"] - TARGET_COVERAGE).abs()
results_df = results_df.sort_values(
    ["coverage_gap", "test_width", "test_mse_mid", "val_pinball_low", "val_pinball_high"],
    ascending=[True, True, True, True, True]
).reset_index(drop=True)

results_df.to_csv(RESULTS_CSV, index=False)

print("\nTop 10 configurations:")
print(results_df.head(10).to_string(index=False))

best_row = results_df.iloc[0].to_dict()
best_key = (
    f"{best_row['feature_set']}|L{int(best_row['lookback'])}|lag{int(best_row['lags'])}|"
    f"{int(best_row['num_layers'])}x{int(best_row['hidden_size'])}|q{best_row['quantile_label']}|{MODEL_KIND}"
)
best_art = artifacts[best_key]

best_pred_df = pd.DataFrame({
    "Date": pd.to_datetime(best_art["dates_test"]),
    "Actual": best_art["y_test"],
    "Lower_PI": best_art["low_test"],
    "Upper_PI": best_art["high_test"],
    "Midpoint": best_art["mid_test"],
})
best_pred_df.to_csv(BEST_PRED_CSV, index=False)

print("\nBest configuration:")
print(pd.Series(best_row).to_string())



===== FEATURE SET: Base_Price_Only =====
Training Base_Price_Only|L30|lag1|1x64|q0.050/0.950|LSTM


C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch

Training Base_Price_Only|L30|lag1|1x64|q0.025/0.975|LSTM


C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:59: FutureWarning: `torch.cuda.amp.a

Training Base_Price_Only|L30|lag1|1x64|q0.010/0.990|LSTM


...

C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:34: FutureWarning: `torch.cuda.amp.a

Training Base_Price_Only|L90|lag1|2x128|q0.050/0.950|LSTM


C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\2554726667.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():


Training Base_Price_Only|L90|lag1|2x128|q0.025/0.975|LSTM


C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\admin\AppData\Local\Temp\ipykernel_17688\1704013579.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [ ]:
def summarize_feature_set(df_res, feature_name):
    sub = df_res[df_res["feature_set"] == feature_name].copy()
    if sub.empty:
        return None
    idx = (sub["coverage_gap"] + 0.001 * sub["test_width"]).idxmin()
    return sub.loc[idx]

base_best = summarize_feature_set(results_df, "Base_Price_Only")
gated_best = summarize_feature_set(results_df, "Semantic_Gated")
ungated_best = summarize_feature_set(results_df, "Semantic_Ungated")

print("\nFeature-set comparison:")
comparison_rows = []
for label, row in [("Base_Price_Only", base_best), ("Semantic_Gated", gated_best), ("Semantic_Ungated", ungated_best)]:
    if row is None:
        continue
    comparison_rows.append({
        "feature_set": label,
        "coverage": row["test_coverage"],
        "width": row["test_width"],
        "mse_mid": row["test_mse_mid"],
        "coverage_gap": row["coverage_gap"],
    })
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

In [ ]:

if base_best is not None and gated_best is not None:
    uplift_cov = gated_best["test_coverage"] - base_best["test_coverage"]
    uplift_width = base_best["test_width"] - gated_best["test_width"]
    print(f"\nSemantic uplift vs base:")
    print(f"Coverage change: {uplift_cov:+.4f}")
    print(f"Width change: {uplift_width:+.4f} (positive means narrower interval)")

if gated_best is not None and ungated_best is not None:
    gate_cov = gated_best["test_coverage"] - ungated_best["test_coverage"]
    gate_width = ungated_best["test_width"] - gated_best["test_width"]
    print(f"\nNews_flag gating effect (gated minus ungated):")
    print(f"Coverage change: {gate_cov:+.4f}")
    print(f"Width change: {gate_width:+.4f} (positive means narrower interval)")


In [ ]:
# ============================================================
# 9. SENSITIVITY HEATMAPS
# ============================================================
# Use the best feature set for sensitivity visualization.
best_feature_set = best_row["feature_set"]
s_df = results_df[results_df["feature_set"] == best_feature_set].copy()

# Heatmap A: Lookback vs Neurons
heat1 = (
    s_df.groupby(["lookback", "hidden_size"], as_index=False)["test_width"]
    .mean()
    .pivot(index="lookback", columns="hidden_size", values="test_width")
    .sort_index()
)

# Heatmap B: Lag vs p-value
heat2 = (
    s_df.groupby(["lags", "quantile_label"], as_index=False)["test_width"]
    .mean()
    .pivot(index="lags", columns="quantile_label", values="test_width")
    .sort_index()
)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

im1 = axes[0].imshow(heat1.values, aspect="auto", origin="lower")
axes[0].set_title(f"Lookback vs Neurons\n(Mean Width, {best_feature_set})")
axes[0].set_xticks(np.arange(len(heat1.columns)))
axes[0].set_xticklabels(heat1.columns.astype(int))
axes[0].set_yticks(np.arange(len(heat1.index)))
axes[0].set_yticklabels(heat1.index.astype(int))
axes[0].set_xlabel("Hidden units")
axes[0].set_ylabel("Lookback")
for i in range(heat1.shape[0]):
    for j in range(heat1.shape[1]):
        axes[0].text(j, i, f"{heat1.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(heat2.values, aspect="auto", origin="lower")
axes[1].set_title(f"Lag vs p-value\n(Mean Width, {best_feature_set})")
axes[1].set_xticks(np.arange(len(heat2.columns)))
axes[1].set_xticklabels(heat2.columns, rotation=30, ha="right")
axes[1].set_yticks(np.arange(len(heat2.index)))
axes[1].set_yticklabels(heat2.index.astype(int))
axes[1].set_xlabel("Quantile pair")
axes[1].set_ylabel("Lag")
for i in range(heat2.shape[0]):
    for j in range(heat2.shape[1]):
        axes[1].text(j, i, f"{heat2.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10. TRADE-OFF PLOTS
# ============================================================
plt.figure(figsize=(12, 5))
for fs_name, sub in results_df.groupby("feature_set"):
    sub = sub.sort_values("test_width")
    plt.plot(sub["test_width"], sub["test_coverage"], marker="o", linewidth=1.2, label=fs_name, alpha=0.85)
plt.axhline(TARGET_COVERAGE, linestyle="--", linewidth=1.2, color="black", label="Target coverage")
plt.xlabel("Mean interval width")
plt.ylabel("Coverage probability")
plt.title("Coverage vs Interval Width across all tweaks")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 11. OPTIONAL ARCHITECTURE COMPARISON
# ============================================================
def evaluate_architecture(model_kind: str, ref_row: pd.Series):
    feature_set_name = ref_row["feature_set"]
    lookback = int(ref_row["lookback"])
    n_lags = int(ref_row["lags"])
    num_layers = int(ref_row["num_layers"])
    hidden_size = int(ref_row["hidden_size"])
    q_low = float(ref_row["q_low"])
    q_high = float(ref_row["q_high"])

    feature_cols = feature_sets[feature_set_name]
    lagged_df = add_lag_features(df, feature_cols, n_lags)
    lagged_feature_cols = feature_cols + [f"{col}_lag{k}" for col in feature_cols for k in range(1, n_lags + 1)]
    X_seq, y_seq, dates_seq = create_sequences(lagged_df, lagged_feature_cols, lookback)
    splits = chronological_split(X_seq, y_seq, dates_seq, train_frac=0.70, val_frac=0.15)
    X_train, y_train, _ = splits["train"]
    X_val, y_val, _ = splits["val"]
    X_test, y_test, d_test = splits["test"]

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    x_scaler.fit(X_train.reshape(-1, X_train.shape[-1]))
    y_scaler.fit(y_train.reshape(-1, 1))

    def transform_X(X):
        X2 = X.reshape(-1, X.shape[-1])
        return x_scaler.transform(X2).reshape(X.shape).astype(np.float32)

    X_train_s = transform_X(X_train)
    X_val_s = transform_X(X_val)
    X_test_s = transform_X(X_test)

    y_train_s = y_scaler.transform(y_train.reshape(-1, 1)).ravel().astype(np.float32)
    y_val_s = y_scaler.transform(y_val.reshape(-1, 1)).ravel().astype(np.float32)

    low_model, high_model, _, _ = fit_quantile_pair(
        X_train_s, y_train_s, X_val_s, y_val_s,
        input_size=X_train_s.shape[-1],
        hidden_size=hidden_size,
        num_layers=num_layers,
        model_kind=model_kind,
        q_low=q_low,
        q_high=q_high,
    )

    low_val = y_scaler.inverse_transform(predict_numpy(low_model, X_val_s).reshape(-1, 1)).ravel()
    high_val = y_scaler.inverse_transform(predict_numpy(high_model, X_val_s).reshape(-1, 1)).ravel()
    scale, _ = calibrate_symmetric_interval(y_val, low_val, high_val, TARGET_COVERAGE)

    low_test = y_scaler.inverse_transform(predict_numpy(low_model, X_test_s).reshape(-1, 1)).ravel()
    high_test = y_scaler.inverse_transform(predict_numpy(high_model, X_test_s).reshape(-1, 1)).ravel()
    low_test_cal, high_test_cal = apply_interval_scale(low_test, high_test, scale)

    cov, width, mse = interval_metrics(y_test, low_test_cal, high_test_cal)
    cov_ci, width_ci = bootstrap_ci(y_test, low_test_cal, high_test_cal, reps=BOOTSTRAP_REPS, seed=RANDOM_SEED)

    return {
        "model_kind": model_kind,
        "coverage": cov,
        "width": width,
        "mse_mid": mse,
        "cov_ci_low": cov_ci[0],
        "cov_ci_high": cov_ci[1],
        "width_ci_low": width_ci[0],
        "width_ci_high": width_ci[1],
        "dates_test": d_test,
        "y_test": y_test,
        "low_test": low_test_cal,
        "high_test": high_test_cal,
    }

In [ ]:

arch_rows = []
arch_artifacts = {}
for arch in ARCHITECTURES_TO_COMPARE:
    print(f"Evaluating architecture: {arch}")
    arch_res = evaluate_architecture(arch, best_row)
    arch_rows.append({
        "architecture": arch_res["model_kind"],
        "coverage": arch_res["coverage"],
        "width": arch_res["width"],
        "mse_mid": arch_res["mse_mid"],
        "coverage_ci_low": arch_res["cov_ci_low"],
        "coverage_ci_high": arch_res["cov_ci_high"],
        "width_ci_low": arch_res["width_ci_low"],
        "width_ci_high": arch_res["width_ci_high"],
    })
    arch_artifacts[arch] = arch_res

arch_df = pd.DataFrame(arch_rows)
print("\nArchitecture comparison:")
print(arch_df.to_string(index=False))

# Error-bar bar chart: coverage
plt.figure(figsize=(10, 5))
x = np.arange(len(arch_df))
cov_err = np.vstack([
    arch_df["coverage"] - arch_df["coverage_ci_low"],
    arch_df["coverage_ci_high"] - arch_df["coverage"],
])
plt.bar(x, arch_df["coverage"], yerr=cov_err, capsize=5, alpha=0.85)
plt.xticks(x, arch_df["architecture"])
plt.axhline(TARGET_COVERAGE, linestyle="--", linewidth=1.2, color="black", label="Target coverage")
plt.ylabel("Coverage probability")
plt.title("Architecture comparison: coverage")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:


# ============================================================
# 0. GLOBAL CONFIGURATION
# ============================================================





# ============================================================
# 1. DEVICE / SEEDS
# ============================================================


# ============================================================
# 2. DATA LOADING + FEATURE ENGINEERING
# ============================================================









# ============================================================

# ============================================================


# ============================================================
# 4. LOSS + TRAINING HELPERS
# ============================================================







# ============================================================
# 5. CALIBRATION LOGIC
# ============================================================






# ============================================================
# 6. FEATURE SETS
# ============================================================

# ============================================================
# 7. GRID SEARCH
# ============================================================

# ============================================================
# 8. COMPARATIVE ANALYSIS
# ============================================================







# Error-bar bar chart: width
plt.figure(figsize=(10, 5))
width_err = np.vstack([
    arch_df["width"] - arch_df["width_ci_low"],
    arch_df["width_ci_high"] - arch_df["width"],
])
plt.bar(x, arch_df["width"], yerr=width_err, capsize=5, alpha=0.85)
plt.xticks(x, arch_df["architecture"])
plt.ylabel("Mean interval width")
plt.title("Architecture comparison: interval width")
plt.tight_layout()
plt.show()

# ============================================================
# 12. BEST FORECAST VISUAL
# ============================================================
plt.figure(figsize=(13, 5))
plt.plot(pd.to_datetime(best_art["dates_test"]), best_art["y_test"], label="Actual", linewidth=1.5)
plt.plot(pd.to_datetime(best_art["dates_test"]), best_art["mid_test"], label="Midpoint forecast", linewidth=1.2)
plt.fill_between(pd.to_datetime(best_art["dates_test"]), best_art["low_test"], best_art["high_test"], alpha=0.20, label="95% interval")
plt.title(
    f"Best model forecast: {best_row['feature_set']} | L={int(best_row['lookback'])} | lag={int(best_row['lags'])} | "
    f"{int(best_row['num_layers'])}x{int(best_row['hidden_size'])} | q={best_row['quantile_label']}"
)
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 13. FINAL SUMMARY
# ============================================================
print("\nSaved results to:")
print(f" - {RESULTS_CSV}")
print(f" - {BEST_PRED_CSV}")

print("\nBest configuration summary:")
print(
    f"Feature set={best_row['feature_set']}, lookback={int(best_row['lookback'])}, lag={int(best_row['lags'])}, "
    f"layers={int(best_row['num_layers'])}, hidden={int(best_row['hidden_size'])}, q={best_row['quantile_label']}, "
    f"coverage={best_row['test_coverage']:.4f}, width={best_row['test_width']:.4f}, mse_mid={best_row['test_mse_mid']:.4f}"
)